# Alfred: Nemotron baseline

Measures where NVIDIA's Nemotron models fall short as a personal agent, before any training.

**Runtime:** Runtime → Change runtime type → **A100** (an L4 also works). Run the cells in order.

Results are saved to Google Drive (`alfred-results/`), and every run skips episodes it already finished, so if the runtime disconnects, just run the cells again.

What it runs, all on the 45 held-out tasks (5 worlds × 9 task types), 3 tries each:

1. **Nemotron 3 Nano 4B**, the model we'll train, served here with vLLM. Once with reasoning off and once with it on.
2. **Nemotron 3 Nano 30B-A3B** and **Nemotron 3 Super** on Nebius Token Factory, as bigger reference points. This costs well under $1 of the hackathon credits.

Token Factory needs your API key. Add it as a Colab secret: click the key icon in the left sidebar, add a secret named `NEBIUS_API_KEY`, and turn on notebook access. Don't paste the key into a cell.

In [ ]:
# 1. Get the code and results folder ready (about 1 minute)
import glob, os, shlex, subprocess, sys, time, urllib.request
from google.colab import drive

drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/alfred-results'
os.makedirs(f'{OUT}/runs', exist_ok=True)

REPO_URL = ''  # after you push the repo: 'https://github.com/<you>/alfred.git'. Empty: use alfred.zip from your Drive.
if not os.path.exists('/content/alfred/pyproject.toml'):
    if REPO_URL:
        subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/alfred'], check=True)
    else:
        zip_path = '/content/drive/MyDrive/alfred.zip'
        assert os.path.exists(zip_path), 'Upload alfred.zip to the top level of your Google Drive, or set REPO_URL.'
        subprocess.run(['unzip', '-q', '-o', zip_path, '-d', '/content'], check=True)
os.chdir('/content/alfred')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)


def alfred(*args):
    # Runs `python -m alfred.<args>` and streams its output here.
    get_ipython().system('cd /content/alfred && python -m alfred.' + ' '.join(shlex.quote(str(a)) for a in args))


def resume(label):
    # Continue the latest run with this label instead of starting over.
    done = sorted(d for d in glob.glob(f'{OUT}/runs/*-{label}') if os.path.exists(f'{d}/results.jsonl'))
    return ['--resume', done[-1]] if done else ['--out', f'{OUT}/runs', '--label', label]


alfred('eval', '--agent', 'oracle', '--worlds', 1, '--out', '/content/oracle-check')

In [ ]:
# 2. Install vLLM and NVIDIA's reasoning parser for Nemotron (3-5 minutes)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'vllm>=0.15.1'], check=True)
PARSER = '/content/nano_v3_reasoning_parser.py'
if not os.path.exists(PARSER):
    urllib.request.urlretrieve(
        'https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16/resolve/main/nano_v3_reasoning_parser.py', PARSER)
print('vLLM', subprocess.run(['vllm', '--version'], capture_output=True, text=True).stdout.strip())

In [ ]:
# 3. Serve Nemotron 3 Nano 4B (the first start downloads about 8 GB)
MODEL, SERVED = 'nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16', 'nemotron-3-nano-4b'


def start_vllm():
    log = open('/content/vllm.log', 'w')
    proc = subprocess.Popen(
        ['vllm', 'serve', MODEL, '--served-model-name', SERVED, '--port', '8000',
         '--max-model-len', '32768', '--max-num-seqs', '16', '--trust-remote-code',
         '--mamba_ssm_cache_dtype', 'float32', '--enable-auto-tool-choice', '--tool-call-parser', 'qwen3_coder',
         '--reasoning-parser-plugin', PARSER, '--reasoning-parser', 'nano_v3'],
        stdout=log, stderr=subprocess.STDOUT)
    deadline = time.time() + 20 * 60
    while time.time() < deadline and proc.poll() is None:
        try:
            urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5)
            print('vLLM is serving', SERVED)
            return proc
        except Exception:
            time.sleep(5)
    print(open('/content/vllm.log').read()[-4000:])
    raise RuntimeError("vLLM didn't start. The end of its log is above.")


try:
    urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5)
    print('vLLM is already running')
except Exception:
    server = start_vllm()

In [ ]:
# 4. Smoke test: 9 tasks, native tool calling vs JSON tool calls. Picks whichever works better.
COMMON = ['--models', SERVED, '--base-url', 'vllm', '--temperature', 0.6, '--extra-body', '{"top_p": 0.95}',
          '--workers', 8]
for mode in ('native', 'json'):
    alfred('eval', *COMMON, '--no-think', '--world-start', 1100, '--worlds', 1, '--tool-mode', mode,
           *resume(f'smoke-nemotron4b-{mode}'))
pick = subprocess.run([sys.executable, '-m', 'alfred.capability', 'pick-mode', f'{OUT}/runs', 'smoke-nemotron4b'],
                      capture_output=True, text=True, cwd='/content/alfred')
MODE = pick.stdout.strip() or 'native'
print(pick.stderr.strip())
print('Using', MODE, 'tool calls')

In [ ]:
# 5. Main runs: 45 tasks x 3 tries, reasoning off, then on (roughly 10-30 minutes)
alfred('eval', *COMMON, '--tool-mode', MODE, '--worlds', 5, '--samples', 3, '--no-think',
       *resume('main-nemotron4b-nothink'))
alfred('eval', *COMMON, '--tool-mode', MODE, '--worlds', 5, '--samples', 3, '--max-tokens', 8192,
       *resume('main-nemotron4b-think'))

In [ ]:
# 6. Bigger Nemotron models on Nebius Token Factory (key from the NEBIUS_API_KEY secret; about 15-30 minutes)
from google.colab import userdata
from openai import OpenAI

try:
    os.environ['NEBIUS_API_KEY'] = userdata.get('NEBIUS_API_KEY')
except Exception:
    import getpass
    os.environ['NEBIUS_API_KEY'] = getpass.getpass('Nebius Token Factory API key (input is hidden): ')

tf = OpenAI(base_url='https://api.tokenfactory.nebius.com/v1/', api_key=os.environ['NEBIUS_API_KEY'])
available = sorted(m.id for m in tf.models.list() if 'nemotron' in m.id.lower())
print('Nemotron models on Token Factory:', *available, sep='\n  ')
WANTED = ['nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B', 'nvidia/nemotron-3-super-120b-a12b']
TF_MODELS = [m for m in WANTED if m in available]
print('Running:', TF_MODELS or 'none of', WANTED, '(edit WANTED to pick from the list above)')
for m in TF_MODELS:
    slug = m.split('/')[-1].lower()
    alfred('eval', '--models', m, '--base-url', 'tokenfactory', '--worlds', 5, '--samples', 3, '--temperature', 0.6,
           '--max-tokens', 8192, '--workers', 4, *resume(f'tf-{slug}'))

In [ ]:
# 7. Report and explorer, next to the re-graded September baseline (GPT-5.4, qwen3, gpt-oss)
new_runs = sorted(d for d in glob.glob(f'{OUT}/runs/*') if ('-main-' in d or '-tf-' in d)
                  and os.path.exists(f'{d}/results.jsonl'))
september = sorted(glob.glob('/content/alfred/results/2026-09-baseline/*/'))
alfred('report', *new_runs, *september, '--out', f'{OUT}/report.md')
alfred('viewer', *new_runs, '--out', f'{OUT}/explorer.html')
get_ipython().system(f'cd {shlex.quote(OUT)} && rm -f results.zip && zip -qr results.zip runs report.md explorer.html')
print('Download alfred-results/results.zip from your Drive and send it over.')